# Autonomous Delivery Fleet — Project Notebook

INSY 3010, Fall 2026. Read the companion assignment, complete all four tasks, and submit this notebook with your reflection.

## Notebook Execution and Setup

A notebook runs cells individually and remembers variables and functions from earlier executions. A Python script (`.py`) normally runs from top to bottom in a fresh process. Python is the language; the interpreter executes your code. For this project, stay in the notebook: restart the runtime and run all cells before saving your final submission, so results do not depend on leftover state. No Anaconda installation or environment-setup deliverable is required.

The setup opens `conn`. Task functions use that connection and leave it open. The final cleanup cell closes it.

The setup uses the bundled `data/fleet.db` or downloads it when needed.

In [ ]:
import csv
import sqlite3
from pathlib import Path
from urllib.request import urlretrieve

db_path = Path("data/fleet.db")
if not db_path.exists():
    db_path.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(
        "https://raw.githubusercontent.com/olearydj/INSY3010/main/project/delivery-fleet/data/fleet.db",
        db_path,
    )
# Re-running setup closes the previous connection before opening another.
if "conn" in globals():
    conn.close()
conn = sqlite3.connect(db_path)
print("Database:", db_path)

## Explore the Database

Investigate the tables, keys, relationships, and records. Use this space to develop the evidence for the database-structure section of your reflection.

In [ ]:
# Explore the database structure.

In [ ]:
# Examine records and investigate features that affect your reports.

## Worked Example — Parameterized Query

This example returns raw mission measurements for a supplied status. The function reuses the query with different values; the parameter tuple passes each value safely. The input connection belongs to the caller, so this function does not close it.

In [ ]:
def missions_for_status(conn, status):
    cur = conn.cursor()
    cur.execute("""SELECT MissionID, DistanceKm, EnergyUsedWh
        FROM DeliveryMission WHERE Status = ? ORDER BY MissionID""", (status,))
    return cur.fetchall()

completed_missions = missions_for_status(conn, "Completed")
print("Completed missions:", len(completed_missions))
print("First three:", completed_missions[:3])
print("Unknown status:", missions_for_status(conn, "Unknown"))

## Task 1 — Completed Deliveries by Depot

Read the provided Python implementation. Write the SQL implementation below it. Return `(depot_name, completed_count, total_distance_km)` tuples for Completed missions only. Sort by count descending, then name ascending; round the summed distance to two decimal places. SQL must do the filtering, aggregation, and ordering. See the assignment for full details.

In [ ]:
# PROVIDED — Python implementation
def completed_by_depot_python(conn):
    cur = conn.cursor()
    cur.execute('''SELECT d.DepotID, d.Name, m.Status, m.DistanceKm
        FROM Depot d JOIN Robot r ON d.DepotID = r.DepotID
        JOIN DeliveryMission m ON r.RobotID = m.RobotID''')
    stats = {}
    for depot_id, name, status, distance in cur.fetchall():
        if status == 'Completed':
            if depot_id not in stats:
                stats[depot_id] = [name, 0, 0.0]
            stats[depot_id][1] += 1
            stats[depot_id][2] += distance
    result = []
    for name, count, distance in stats.values():
        result.append((name, count, round(distance, 2)))
    def by_name(row):
        return row[0]
    def by_count(row):
        return row[1]
    result.sort(key=by_name)
    result.sort(key=by_count, reverse=True)
    return result

In [ ]:
def completed_by_depot_sql(conn):
    # YOUR CODE HERE: execute one reporting query and return its rows.
    return None

Run this cell after implementing Task 1 to compare your SQL report with the supplied Python report. It checks depot names, counts, distances, and ordering.

In [ ]:
python_report = completed_by_depot_python(conn)
sql_report = completed_by_depot_sql(conn)
if sql_report is None:
    print("Task 1: write your implementation, then rerun this comparison.")
else:
    assert len(sql_report) == len(python_report), "Different number of depots."
    for actual, expected in zip(sql_report, python_report):
        assert actual[:2] == expected[:2], "Check depot names, counts, and ordering."
        assert abs(actual[2] - expected[2]) < 0.005, "Check total distance."
    print("Task 1: implementations agree on this database.")
    for row in sql_report:
        print(row)

## Task 2 — Failed Missions by Zone

Read the supplied SQL implementation. Implement Python filtering, grouping, counting, summing, and sorting after fetching raw joined rows. Do not use SQL WHERE, GROUP BY, HAVING, DISTINCT, ORDER BY, or aggregate functions in your version. Return `(zone_name, failed_count, failed_distance_km)` tuples ordered by count descending, then name ascending. Count zero-distance failures; omit zones without failures.

In [ ]:
# PROVIDED — SQL implementation
def failed_by_zone_sql(conn):
    cur = conn.cursor()
    cur.execute('''SELECT z.Name, COUNT(*) AS FailedCount,
        ROUND(SUM(m.DistanceKm), 2) AS FailedDistanceKm
        FROM DeliveryZone z JOIN DeliveryMission m ON z.ZoneID = m.ZoneID
        WHERE m.Status = 'Failed'
        GROUP BY z.ZoneID, z.Name
        ORDER BY FailedCount DESC, z.Name ASC''')
    return cur.fetchall()

In [ ]:
def failed_by_zone_python(conn):
    # YOUR CODE HERE: fetch raw joined rows and process them in Python.
    return None

Run this cell after implementing Task 2 to compare your Python report with the supplied SQL report. It checks zone names, failure counts, distances, and ordering.

In [ ]:
sql_failures = failed_by_zone_sql(conn)
python_failures = failed_by_zone_python(conn)
if python_failures is None:
    print("Task 2: write your implementation, then rerun this comparison.")
else:
    assert len(python_failures) == len(sql_failures), "Different number of zones."
    for actual, expected in zip(python_failures, sql_failures):
        assert actual[:2] == expected[:2], "Check zone names, counts, and ordering."
        assert abs(actual[2] - expected[2]) < 0.005, "Check failed distance."
    print("Task 2: implementations agree on this database.")
    for row in python_failures:
        print(row)

## Task 3 — CSV Export

Complete the supplied `try` block. Use `csv.writer` and `with open(...)`, write headers followed by data, and return `True`. The provided file-access error branch returns `False`. File I/O and CSV will be covered in Files & Exceptions. Read back actual field values to verify your output, including commas and quotes. Do not modify the database.

In [ ]:
def export_to_csv(data, headers, filename):
    try:
        # YOUR CODE HERE: open with newline="" and encoding="utf-8".
        # Write the header, write the rows, then return True.
        return None
    except OSError as error:
        print(f"Could not write {filename}: {error}")
        return False

Run this cell after implementing Task 3 to write and read back a small CSV containing commas and quotation marks. The assertions check the recovered field values and row order.

In [ ]:
output_dir = Path("project-output")
output_dir.mkdir(exist_ok=True)
example_rows = [("North, Annex", 2), ('Robot "Scout"', 3)]
example_headers = ["Label", "Count"]
example_path = output_dir / "quoting-example.csv"
result = export_to_csv(example_rows, example_headers, example_path)
if result is None:
    print("Task 3: implement export_to_csv, then rerun this example.")
else:
    assert result is True
    with open(example_path, newline="", encoding="utf-8") as f:
        restored = list(csv.reader(f))
    assert restored == [["Label", "Count"], ["North, Annex", "2"], ['Robot "Scout"', "3"]]
    print("CSV field values, quotes, commas, and order match.")

Use another code cell to export your actual Task 1 report, read it back, and compare its contents. Also test empty data with headers and a filename inside a parent directory that does not exist. CSV reads numbers as text; account for that when comparing. Record the expected and actual results in your reflection.

In [ ]:
# YOUR CODE HERE: actual report, header-only export, and failed-write evidence.

## Task 4 — Energy Intensity by Depot and Terrain

Use SQL, Python, or both. Return `(depot_name, terrain_type, completed_count, total_distance_km, total_energy_wh, wh_per_km)`. Include Completed missions only. Group by depot and terrain, omit nonexistent groups, and sort by depot name then terrain ascending. Round the three final measurements to two decimals, after computing from unrounded sums.

Calculate Wh/km from each group's total energy and total distance. Explain your approach and verify at least one group independently.

In [ ]:
def energy_by_depot_terrain(conn):
    # YOUR CODE HERE: implement the required report.
    return None


Run this cell to display your Task 4 report. Use the output to choose groups for your own verification.

In [ ]:
energy_report = energy_by_depot_terrain(conn)
if energy_report is None:
    print("Task 4: implement the integrated energy report.")
else:
    for row in energy_report:
        print(row)

## Your Additional Verification

Add cells showing checks beyond the provided comparisons: expected and actual values, an empty or boundary case, and a check that catches a plausible mistake. Do not change `fleet.db`. Review your code against the required division of work between SQL and Python.

In [ ]:
# YOUR CODE HERE: additional checks and intermediate results for the reflection.

## Clean Up

Close the connection after completing your work. Save the notebook with relevant output and submit it alongside the reflection.

In [ ]:
conn.close()
print("Database connection closed.")

---

Auburn University / Industrial and Systems Engineering  
INSY 3010 / Programming and Databases for ISE / Fall 2026  
© Copyright 2026, Danny J. O'Leary.